# Modeling

**Goal:** Train baseline models and compare their performance on predicting purchase intention.

In [1]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from src.preprocessing import load_data, clean_data
from src.models import (
    get_logistic_regression,
    get_decision_tree,
    get_mlp,
    get_catboost
)
from src.evaluation import evaluate_model, cross_validate_model

## 1. Load and Prepare Data

In [2]:
df = load_data()
df = clean_data(df)

print(f"Shape after cleaning: {df.shape}")

X = df.drop(columns=["Revenue"])
y = df["Revenue"]

numerical_features = [
    "Administrative", "Administrative_Duration",
    "Informational", "Informational_Duration",
    "ProductRelated", "ProductRelated_Duration",
    "BounceRates", "ExitRates", "PageValues", "SpecialDay"
]

categorical_features = [
    "Month", "OperatingSystems", "Browser", "Region",
    "TrafficType", "VisitorType", "Weekend"
]

Shape after cleaning: (12205, 18)


## 2. Train / Test Split

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: (9764, 17), Test: (2441, 17)


## 3. Initialize models

In [4]:
models = {
    "Logistic Regression": get_logistic_regression(numerical_features, categorical_features),
    "Decision Tree": get_decision_tree(numerical_features, categorical_features),
    "MLP": get_mlp(numerical_features, categorical_features),
    "CatBoost": get_catboost()
}

## 4. Cross-validation

In [5]:
cv_results = []

for name, model in models.items():
    if name == "CatBoost":
        result = cross_validate_model(
            model, X_train, y_train, name,
            cat_features=categorical_features
        )
    else:
        result = cross_validate_model(model, X_train, y_train, name)
    
    cv_results.append(result)

pd.DataFrame(cv_results)

,Model,Accuracy,F1-Score,ROC-AUC
0,Logistic Regression,0.8471,0.6066,0.8991
1,Decision Tree,0.8354,0.6119,0.8915
2,MLP,0.8984,0.6471,0.9173
3,CatBoost,0.8880,0.6626,0.9222


## 5. Final model evaluation

In [6]:
best_model = get_catboost()
best_model.fit(X_train, y_train, cat_features=categorical_features)

pd.DataFrame([evaluate_model(best_model, X_test, y_test, "CatBoost")])

,Model,Accuracy,F1-Score,ROC-AUC
0,CatBoost,0.8865,0.6706,0.9227


## 6. Feature Importance

In [7]:
importance = best_model.get_feature_importance()
feature_names = X_train.columns

pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance
}).sort_values("Importance", ascending=False)

,Feature,Importance
10,Month,21.286890
8,PageValues,14.813571
7,ExitRates,9.475556
5,ProductRelated_Duration,9.159932
4,ProductRelated,7.227061
6,BounceRates,6.068674
1,Administrative_Duration,5.330485
14,TrafficType,5.288512
0,Administrative,4.540720
13,Region,3.619449


## 7. Conclusion

Cross-validation identified CatBoost as the strongest model.  
Final evaluation on the held-out test set confirms its performance.